<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **2. PID Controller**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 2.1 Introduction

We already explored the fundamentals of control systems - how sensors, actuators, and controllers interact in open and closed loops to make systems behave in a desired way. One of the simplest and most widely used mechanisms for achieving this is the **PID controller** (Proportional-Integral-Derivative controller). Its role is to continuously adjust the behavior of a system so that it reaches and maintains a desired target state (the setpoint).

### 2.2 Feedback Loop

A control system continuously measures the output of a process, compares it to a desired reference, and adjusts its input to reduce any difference.
This closed-loop interaction between sensor, controller, and actuator allows systems to maintain stability and desired performance even when conditions change.

<center>
<img src="../../.admin/notebooks/picture/closed_loop.png" 
     alt="Closed Loop Control System" 
     width="50%" 
     style="filter: grayscale(60%);">
<p align="center"><em>Feedback (or, closed-loop) system</em></p>
</center>


We can now define PID control as a form of closed-loop control that calculates its system input from proportional, integral, and derivative terms based on the measured output. To understand this better, let’s use a real example: **car cruise control**. Imagine you’re driving on the highway and you want the car to stay at a constant speed (say, 100 km/h).

- The setpoint is your desired speed.
- The sensor is the speedometer, measuring current speed.
- The controller adjusts throttle or brakes.
- The system is the car (engine + drag + road).

The controller’s job: keep the actual speed as close as possible to the target.

#### 2.2.1 Proportional Control

The simplest form of control is **proportional (P)** control: If the car is below the target speed, the controller (the driver) increases throttle. If it is above the target, the controller reduces throttle. The size of the correction is proportional to the error (the difference between desired and measured speed). Use the slider below to change the proportional gain $K_p$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def simulate_p_response(Kp=1.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 10.0    # desired final position
    setpoint_time = 10.0     # time when setpoint jumps

    x = 0.0                  # position
    v = 0.0                  # velocity
    values, setpoints = [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        u = Kp * error

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        setpoints.append(setpoint)

    # Plot the response
    plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.legend()
    plt.grid(True)
    plt.show()

# Interactive widget
interact(
    simulate_p_response,
    Kp=widgets.FloatSlider(value=1.0, min=0.0, max=25.0, step=0.1),
)


With a small P, the car reacts only weakly to the speed error. This means the extra throttle added is quickly balanced by drag, and the car settles below the target speed. With a larger P, the controller pushes harder, so the car gets closer to the target. But the stronger reactions often cause the car to overshoot - it goes above the target, then corrects back down, and this cycle leads to oscillations. With a very large P, the corrections become so aggressive that the car destabilizes; it keeps swinging around the target speed without ever calming down.

Proportional control is like a driver glancing at the speedometer: If a driver presses the pedal only gently, drag wins and the car never reaches the desired speed. If the driver presses harder, the car overshoots and then has to back off, creating a back-and-forth wobble. Finally, if the driver reacts too aggressively, the car over-corrects every time, and the system never stabilizes.

To fix this we add another component: the **integral term**.


#### 2.2.2 Proportional + Integral Control

The integral looks at the accumulated error over time. If the car is always a bit slower than the target, the integral grows and slowly adds more throttle until the error disappears. Once the speed matches the target, the error stops accumulating, and the correction levels off. In other words: the integral term remembers past error and pushes the car to eliminate it completely. Use the slider below to adjust the integral gain $K_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

def simulate_pid_response(Kp=1.0, Ki=0.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 100.0   # desired final value
    setpoint_time = 10.0     # time when setpoint jumps

    x = 0.0                  # position
    v = 0.0                  # velocity
    integral = 0.0
    values, errors, integrals, setpoints = [], [], [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        integral += error * dt
        u = Kp * error + Ki * integral

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        errors.append(error)
        integrals.append(integral)
        setpoints.append(setpoint)

    # Plot the response
    plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    simulate_pid_response,
    Kp=widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1),
    Ki=widgets.FloatSlider(value=0.0, min=0.0, max=0.2, step=0.001),
)


The integral term removes the steady-state error. But if it is too strong, it can also make oscillations worse, since the controller may keep “pushing” even after the error starts changing sign. Yet, the integral term is unable to remove the oscillations. To handle that, we need one more component: the **derivative term**,  leading us to the full PID controller.

#### 2.2.3 Proportional + Integral + Derivative Control

The derivative looks at how quickly the error is changing. If the car is approaching the target speed too fast, the derivative pushes back and slows the correction to avoid overshoot. If the car is slowing down too quickly, the derivative eases the adjustment to prevent undershoot. In other words: the derivative term anticipates the future trend and helps reduce oscillations, making the system smoother. Use the slider below to adjust the derivative gain $K_d$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

def simulate_pid_response(Kp=1.0, Ki=0.0, Kd=0.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 100.0  # desired final value
    setpoint_time = 10.0    # time when setpoint jumps
    x = 0.0                 # position
    v = 0.0                 # velocity
    integral = 0.0
    previous_error = 0.0
    values, errors, integrals, setpoints = [], [], [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        integral += error * dt
        derivative = (error - previous_error) / dt
        u = Kp * error + Ki * integral + Kd * derivative

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        errors.append(error)
        integrals.append(integral)
        setpoints.append(setpoint)
        previous_error = error

    # Plot the response
    plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.legend()
    plt.grid(True)
    plt.show()

# Interactive widget
interact(
    simulate_pid_response,
    Kp=widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1),
    Ki=widgets.FloatSlider(value=0.08, min=0.0, max=0.2, step=0.001),
    Kd=widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1),
)


### 2.3 PID Controller

Together, the three terms form the **PID controller**:  

- **P** reacts to the current error.  
- **I** removes long-term bias by accumulating past error.  
- **D** anticipates future error and reduces oscillations.  

By tuning $(K_p, K_i, K_d)$, we can shape how the system behaves - whether it should react quickly, settle smoothly, or resist disturbances. This balance between **speed, accuracy, and stability** is why PID controllers are used everywhere, from cars and drones to mobile systems and network control.  


#### 2.3.1 PID Block Diagram  

<center>
<img src="../../.admin/notebooks/picture/pid_en.png" 
     alt="PID Block Diagram" 
     width="70%" 
     style="filter: grayscale(60%);">
<p style="text-align: center; font-size: 0.9em;">
</p>
<p style="text-align: center; font-size: 0.9em;">
Image source: <a href="https://commons.wikimedia.org/wiki/File:PID_en.svg">Wikimedia Commons</a> · 
License: <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>
</p>
<p align="center"><em>PID Block Diagram</em></p>
</center>

#### 2.3.2 PID Equation  

The controller output is calculated as:  

$$
u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\, d\tau + K_d \frac{de(t)}{dt}
$$

where:  
- $e(t)$ = setpoint − measured value (the error)  
- $K_p, K_i, K_d$ = proportional, integral, derivative gains  

#### 2.3.3 Regulatory Acceptance

PID controllers are considered deterministic systems, which is one of the main reasons they are trusted and accepted in medical devices and other safety-critical technologies. A PID controller produces an output that depends directly and transparently on measurable quantities: the current error, the accumulated error over time, and the rate of change of the error. In other words, for any given sequence of inputs, the controller’s output will always be the same. There is no randomness, no adaptive learning process, and no hidden internal dynamics that might cause unexpected behavior. This determinism makes the controller fully predictable and mathematically analyzable.

Another important reason for regulatory acceptance is the transparency of the PID design. The controller has only three parameters—proportional, integral, and derivative gains—and each has a clear physical meaning. Engineers can explain exactly why each parameter has a certain value, how it was tuned, and what effect it has on the system. There is no “black box” behavior, which makes validation, documentation, and certification straightforward.

When a PID controller malfunctions or is poorly tuned, its failure modes are also well understood: the system may oscillate, overshoot, or drift, but it will not exhibit chaotic or unpredictable responses. This predictability allows additional safety mechanisms and fault detection systems to be built around it. 

For these reasons, regulatory agencies generally favor deterministic and analyzable control systems. PID controllers remain a standard solution in many critical applications, such as insulin pumps and artificial pancreas systems, mechanical ventilators, infusion pumps, and industrial process controls. Even though more advanced control or learning methods exist, PID controllers are still preferred wherever safety, reliability, and traceability are more important than optimization. They are accepted because they are deterministic, interpretable, and their behavior can be rigorously proven - a combination that makes regulatory approval possible.

----
### 2.4 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Which values for the three gains did you select in the interactive example above?** Answer:

(add your answer here)

**What happens to the car’s speed if you set $K_p$ very small?** Answer:

(add your answer here)

**What do you observe when $K_p$ is very large?** Answer:

(add your answer here)

**How does the integral term fix the main weakness of proportional control?** Answer:

(add your answer here)

**Why can a very large $K_i$ make the car overshoot and oscillate?** Answer:

(add your answer here)

**How does the derivative term improve the response compared to PI control?** Answer:

(add your answer here)

**In your own words, explain what each term (P, I, D) cares about: present, past, or future error.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)